In [1]:
import os 
from pathlib import Path 
import pandas as pd 
import numpy as np 

import warnings

In [2]:
warnings.filterwarnings("ignore")

In [3]:
os.chdir("..")

In [4]:
print(os.getcwd())

e:\project_archive\student-dropout-enrolled-graduate-rate-prediction


In [6]:
# Load data

data_path = Path("data/processed")
if not data_path.exists():
    raise FileNotFoundError

X_train = pd.read_csv(data_path / "x_train.csv", sep=';')
y_train = np.load(data_path / "y_train.npy")
print("X_train and y_train loaded successfully")

X_test = pd.read_csv(data_path / "x_test.csv", sep=';')
y_test = np.load(data_path / "y_test.npy")
print("X_test and y_test loaded successfully")

X_train and y_train loaded successfully
X_test and y_test loaded successfully


In [7]:
config = {
    "LightGBM": {

        "n_estimators": {
            "min": 200,
            "max": 1000
        },

        "learning_rate": {
            "min": 0.01,
            "max": 0.3,
            "log": True
        },

        "max_depth": {
            "min": 3,
            "max": 10
        },

        "num_leaves": {
            "min": 15,
            "max": 255
        },

        "min_child_samples": {
            "min": 5,
            "max": 100
        },

        "min_child_weight": {
            "min": 1e-3,
            "max": 10.0,
            "log": True
        },

        "subsample": {
            "min": 0.6,
            "max": 1.0
        },

        "subsample_freq": {
            "min": 1,
            "max": 10
        },

        "colsample_bytree": {
            "min": 0.6,
            "max": 1.0
        },

        "reg_alpha": {
            "min": 0.0,
            "max": 10.0
        },

        "reg_lambda": {
            "min": 1e-5,
            "max": 10.0,
            "log": True
        },

        "min_split_gain": {
            "min": 0.0,
            "max": 5.0
        },

        "objective": [
            "multiclass"
        ],

        "boosting_type": [
            "gbdt"
        ],

        "metric": [
            "multi_logloss"
        ],

        "verbosity": [
            -1
        ]
    }
}

In [8]:
def suggest_params(trial, config):

    cfg = config["LightGBM"]

    return {

        "n_estimators": trial.suggest_int(
            "n_estimators",
            cfg["n_estimators"]["min"],
            cfg["n_estimators"]["max"],
            step=50,
        ),

        "learning_rate": trial.suggest_float(
            "learning_rate",
            cfg["learning_rate"]["min"],
            cfg["learning_rate"]["max"],
            log=cfg["learning_rate"]["log"],
        ),

        "max_depth": trial.suggest_int(
            "max_depth",
            cfg["max_depth"]["min"],
            cfg["max_depth"]["max"],
        ),

        "num_leaves": trial.suggest_int(
            "num_leaves",
            cfg["num_leaves"]["min"],
            cfg["num_leaves"]["max"],
        ),

        "min_child_samples": trial.suggest_int(
            "min_child_samples",
            cfg["min_child_samples"]["min"],
            cfg["min_child_samples"]["max"],
        ),

        "min_child_weight": trial.suggest_float(
            "min_child_weight",
            cfg["min_child_weight"]["min"],
            cfg["min_child_weight"]["max"],
            log=cfg["min_child_weight"]["log"],
        ),

        "subsample": trial.suggest_float(
            "subsample",
            cfg["subsample"]["min"],
            cfg["subsample"]["max"],
        ),

        "subsample_freq": trial.suggest_int(
            "subsample_freq",
            cfg["subsample_freq"]["min"],
            cfg["subsample_freq"]["max"],
        ),

        "colsample_bytree": trial.suggest_float(
            "colsample_bytree",
            cfg["colsample_bytree"]["min"],
            cfg["colsample_bytree"]["max"],
        ),

        "reg_alpha": trial.suggest_float(
            "reg_alpha",
            cfg["reg_alpha"]["min"],
            cfg["reg_alpha"]["max"],
        ),

        "reg_lambda": trial.suggest_float(
            "reg_lambda",
            cfg["reg_lambda"]["min"],
            cfg["reg_lambda"]["max"],
            log=cfg["reg_lambda"]["log"],
        ),

        "min_split_gain": trial.suggest_float(
            "min_split_gain",
            cfg["min_split_gain"]["min"],
            cfg["min_split_gain"]["max"],
        ),

        "objective": cfg["objective"][0],
        "boosting_type": cfg["boosting_type"][0],
        "metric": cfg["metric"][0],
        "verbosity": cfg["verbosity"][0],
        "random_state": 42,
        "n_jobs": -1,
    }

In [14]:
import optuna
import wandb
from pprint import pprint

from lightgbm import LGBMClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

def objective(trial, X, y, config, run):

    params = suggest_params(
        trial=trial,
        config=config
    )

    model = LGBMClassifier(
                **params,
          )

    cv_strategy = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=42,
    )

    scores = cross_val_score(
    estimator=model,
    X=X,
    y=y,
    cv=cv_strategy,
    scoring="f1_macro",
    n_jobs=-1,
    error_score="raise",
    )

    mean_score = scores.mean()

    run.log({
        "trial": trial.number,
        "cv_f1_macro": mean_score,
        "cv_f1_macro_std": scores.std(),
    })

    return mean_score

In [15]:
# train.py
from sklearn.metrics import f1_score

with wandb.init(
        project="student-drop-enroll-grad-preds",
        name="LightGBM-Optuna-v2",
        group="LightGBM",
        tags=["LightGBM", "Optuna"]
    ) as run:
     
      study = optuna.create_study(
       direction="maximize",
       study_name="lightgbm_tuning",
    )

      study.optimize(
            lambda trial: objective(
                  trial,
                  X_train,
                  y_train,
                  config=config,
                  run=run
            ),
            n_trials=80
      )

      best_params = study.best_params

      model = LGBMClassifier(
            **best_params,
            objective="multiclass",
            metric="multi_logloss",
            random_state=42,
            n_jobs=-1,
      )

      model.fit(X_train, y_train)

      pred = model.predict(X_test)

      f1 = f1_score(
            y_test,
            pred,
            average='macro'
      )

      run.summary["best_cv_score"] = study.best_value
      run.summary["test_f1_macro"] = f1
      run.summary["best_params"] = best_params


[I 2026-07-31 23:45:24,910] A new study created in memory with name: lightgbm_tuning
[I 2026-07-31 23:45:36,021] Trial 0 finished with value: 0.644621567729909 and parameters: {'n_estimators': 650, 'learning_rate': 0.22675828543729912, 'max_depth': 8, 'num_leaves': 203, 'min_child_samples': 89, 'min_child_weight': 3.7175729898129943, 'subsample': 0.970053687912404, 'subsample_freq': 7, 'colsample_bytree': 0.9092360543115128, 'reg_alpha': 5.141982539889094, 'reg_lambda': 0.01670841719737234, 'min_split_gain': 0.4594131750011754}. Best is trial 0 with value: 0.644621567729909.
[I 2026-07-31 23:45:43,563] Trial 1 finished with value: 0.6160667374979376 and parameters: {'n_estimators': 350, 'learning_rate': 0.020083879610466267, 'max_depth': 9, 'num_leaves': 69, 'min_child_samples': 93, 'min_child_weight': 7.129298605748407, 'subsample': 0.6767660562896045, 'subsample_freq': 10, 'colsample_bytree': 0.8904174082744714, 'reg_alpha': 5.383164784720692, 'reg_lambda': 0.00021000947005613916, 'm

cv_f1_macro,▆▃▃▅▄▁▇▇▇▇▆▂▆█▆▇▇█▇▆▆█▆▇▇▆▇██▇▇██▇▆▆▆█▇▇
cv_f1_macro_std,▃▅▃▄█▅▆▆▁▂▄▂▅▂▅▃▁▅▃▆▃▄▅▅▃▄▄▅▂▆▃▁▅▄▅▄▂▅▄▂
trial,▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇███
best_cv_score,0.67395
cv_f1_macro,0.65964
cv_f1_macro_std,0.01682
test_f1_macro,0.6504
trial,79


In [16]:
study.best_params

{'n_estimators': 200,
 'learning_rate': 0.0813878228331522,
 'max_depth': 5,
 'num_leaves': 121,
 'min_child_samples': 23,
 'min_child_weight': 0.38065697834121526,
 'subsample': 0.9997833996022827,
 'subsample_freq': 6,
 'colsample_bytree': 0.7570918902810027,
 'reg_alpha': 0.04897016843787337,
 'reg_lambda': 0.06909577691783778,
 'min_split_gain': 0.5434142910423587}

In [17]:
study.best_value

0.6739500714665354

In [18]:
import json

save_result_path = Path("artifacts/reports")

save_result_path.mkdir(parents=True, exist_ok=True)

result = {
    "model_name": "LightGBM",
    "best_trial": study.best_trial.number,
    "best_params": study.best_params,
    "best_value": float(study.best_value)
}

with open(save_result_path / "lightgbm.json", "w") as f:
    json.dump(result, f, indent=4)
